# Ingest and Vectorize Zoning Laws

### Tech:

To extract raw text from pdfs use PyPDF2 or PyMuPDF/ for docs: python-docx

Langchain to handle splitting and chunking logic

ChromaDB local vector store in the directory

Embeddings from OpenAI or another service



### Notes:

Semantic Chunking is preferred over Recursive Character splitter (standard) with overlap of some tokens based on semantics. To do that, you need embedding API and then do the semantic chunking

### Read the orlando zoning files

In [19]:
import os
from dotenv import load_dotenv
from docx import Document
from pinecone import Pinecone, ServerlessSpec
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_openai import OpenAIEmbeddings

load_dotenv()

# Load API keys
openai_api_key = os.getenv("OPENAI_API_KEY")
pinecone_api_key = os.getenv("PINECONE_API_KEY")

# Verify API keys are loaded
if not openai_api_key or not pinecone_api_key:
    raise ValueError("OPENAI_API_KEY or PINECONE_API_KEY not found in .env")

# Initialize clients
pc = Pinecone(api_key=pinecone_api_key)

print("✓ API keys loaded successfully")

✓ API keys loaded successfully


In [12]:
orlando_zoning_laws_path = "../data/orlando-zoning-laws"
files = os.listdir(orlando_zoning_laws_path)
for f in files:
    print(f)

Chapter_14___PROPERTY_MAINTENANCE_CODE.docx
Chapter_58___ZONING_DISTRICTS_AND_USES.docx
Chapter_60___SUBDIVISION_AND_LANDSCAPING.docx
Chapter_61___ROADWAY_DESIGN_AND_ACCESS_MANAGEMENT.docx
Chapter_63___ENVIRONMENTAL_PROTECTION.docx
Chapter_65___OFFICERS__BOARDS_AND_PROCEDURES.docx
Chapter_66___DEFINITIONS.docx
Chapter_67___AFFORDABLE_HOUSING.docx
Orlando-Zoning-Chapter-Descriptions.xlsx
OrlandoFLCodeofOrdinancesEXPORT20251209.xlsx


In [13]:
# Get only .docx files
docx_files = [f for f in os.listdir(orlando_zoning_laws_path) if f.endswith('.docx')]

# Dictionary to store text from each document
documents = {}

for filename in docx_files:
    filepath = os.path.join(orlando_zoning_laws_path, filename)
    doc = Document(filepath)
    
    # Extract all paragraphs
    text = "\n".join([para.text for para in doc.paragraphs])
    documents[filename] = text
    
    print(f"✓ Loaded: {filename} ({len(text)} characters)")

print(f"\nTotal documents loaded: {len(documents)}")

✓ Loaded: Chapter_14___PROPERTY_MAINTENANCE_CODE.docx (66196 characters)
✓ Loaded: Chapter_58___ZONING_DISTRICTS_AND_USES.docx (595020 characters)
✓ Loaded: Chapter_60___SUBDIVISION_AND_LANDSCAPING.docx (110424 characters)
✓ Loaded: Chapter_61___ROADWAY_DESIGN_AND_ACCESS_MANAGEMENT.docx (171587 characters)
✓ Loaded: Chapter_63___ENVIRONMENTAL_PROTECTION.docx (98777 characters)
✓ Loaded: Chapter_65___OFFICERS__BOARDS_AND_PROCEDURES.docx (445724 characters)
✓ Loaded: Chapter_66___DEFINITIONS.docx (175223 characters)
✓ Loaded: Chapter_67___AFFORDABLE_HOUSING.docx (58442 characters)

Total documents loaded: 8


### Split and Chunk the orlando zoning laws

In [14]:
splitter = RecursiveCharacterTextSplitter(
    chunk_size=1000,
    chunk_overlap=200,
    separators=["\n\n", "\n", ". ", " ", ""]
)

# Chunk all documents
all_chunks = []

for filename, text in documents.items():
    chunks = splitter.create_documents([text])
    for i, chunk in enumerate(chunks):
        chunk.metadata = {
            "source": filename,
            "chunk_id": i,
            "total_chunks": len(chunks),
            "char_count": len(chunk.page_content),
            "chapter": filename.split("___")[0].replace("_", " "),
        }
    all_chunks.extend(chunks)
    print(f"✓ {filename}: {len(chunks)} chunks")

print(f"\nTotal chunks: {len(all_chunks)}")

✓ Chapter_14___PROPERTY_MAINTENANCE_CODE.docx: 107 chunks
✓ Chapter_58___ZONING_DISTRICTS_AND_USES.docx: 907 chunks
✓ Chapter_60___SUBDIVISION_AND_LANDSCAPING.docx: 171 chunks
✓ Chapter_61___ROADWAY_DESIGN_AND_ACCESS_MANAGEMENT.docx: 257 chunks
✓ Chapter_63___ENVIRONMENTAL_PROTECTION.docx: 138 chunks
✓ Chapter_65___OFFICERS__BOARDS_AND_PROCEDURES.docx: 673 chunks
✓ Chapter_66___DEFINITIONS.docx: 232 chunks
✓ Chapter_67___AFFORDABLE_HOUSING.docx: 89 chunks

Total chunks: 2574


In [15]:
# Check a sample chunk
print(all_chunks[2570].page_content[:500])
print("\n--- Metadata ---")
print(all_chunks[2570].metadata)

Sec. 67.701. Affordable Housing Trust Fund Administration.
The Affordable Housing Trust Fund shall be administered in accordance with the policies and procedures established in Section 193.1 or other applicable policy within the Policies and Procedures Manual for the City of Orlando, as may be amended from time to time. 
(Ord. No. 2023-23, § 1, 9-11-2023, Doc. #2309111201)

Sec. 67.702. Annual Budget Requirements.
In approving the City's annual budget, the City Council shall have the absolute di

--- Metadata ---
{'source': 'Chapter_67___AFFORDABLE_HOUSING.docx', 'chunk_id': 85, 'total_chunks': 89, 'char_count': 733, 'chapter': 'Chapter 67'}


### Generate Embeddings using OpenAI and Pinecone as vector store

In [16]:
# Recommended - explicitly pass API key
embeddings = OpenAIEmbeddings(
    model="text-embedding-3-small",
    openai_api_key=openai_api_key
)

# Test it
# test_vector = embeddings.embed_query("zoning regulations in Orlando")
# print(f"Vector dimensions: {len(test_vector)}")

In [ ]:

# Create a serverless index (free tier)
# pc.create_index(
#     name="orlando-zoning-index",
#     dimension=1536,  # text-embedding-3-small outputs 1536 dims
#     metric="cosine",
#     spec=ServerlessSpec(cloud="aws", region="us-east-1")
# )

{
    "name": "orlando-zoning-index",
    "metric": "cosine",
    "host": "orlando-zoning-index-ndjlfph.svc.aped-4627-b74a.pinecone.io",
    "spec": {
        "serverless": {
            "region": "us-east-1",
            "cloud": "aws",
            "read_capacity": {
                "mode": "OnDemand",
                "status": {
                    "state": "Ready",
                    "current_shards": null,
                    "current_replicas": null
                }
            }
        }
    },
    "status": {
        "ready": true,
        "state": "Ready"
    },
    "vector_type": "dense",
    "dimension": 1536,
    "deletion_protection": "disabled",
    "tags": null,
    "_response_info": {
        "raw_headers": {
            "content-type": "application/json",
            "vary": "origin, access-control-request-method, access-control-request-headers",
            "access-control-allow-origin": "*",
            "access-control-expose-headers": "*",
            "x-pinecone-

In [ ]:
# Connect to the index
index = pc.Index("orlando-zoning-index")

# Upsert


In [23]:
#Retrieval Check
question = "What is the Affordable Housing Trust Fund?"

# Convert question to embedding
query_vector = embeddings.embed_query(question)

# Query Pinecone for similar chunks
results = index.query(
    vector=query_vector,
    top_k=3,
    include_metadata=True
)

# Print results
for match in results['matches']:
    print(f"Score: {match['score']:.4f}")
    print(f"Text: {match['metadata'].get('text', 'N/A')[:200]}...")
    print(f"Source: {match['metadata'].get('source', 'N/A')}")
    print("-" * 50)